# FIFA World Cup 2026 — Monte Carlo Simulation
## Notebook 05: Tournament Simulation (10,000 runs)

**Approach:** Simulate the full 104-match tournament 10,000 times.
Each match result is sampled from XGBoost probabilities.
Penalty shootouts modeled for knockout draws.

**Outputs:**
- P(classify from group) per team
- P(reach R16 / QF / SF / Final / Win) per team
- FiveThirtyEight-style probability table
- Group stage expected standings

In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
import numpy as np
import joblib
import json
from tqdm import tqdm

from world_cup_2026.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR, RANDOM_SEED
)

np.random.seed(RANDOM_SEED)

# ── Load model ─────────────────────────────────────────────────────────────────
MODELS_DIR = Path("../..").resolve() / "models"
xgb_model = joblib.load(MODELS_DIR / "xgb_match_predictor.pkl")

with open(MODELS_DIR / "model_features.json") as f:
    MODEL_FEATURES = json.load(f)

print(f"Model loaded. Features: {len(MODEL_FEATURES)}")

# ── Load fixture ───────────────────────────────────────────────────────────────
df_teams = pd.read_csv(RAW_DATA_DIR / "areezvisram12_fixture" / "teams.csv")
df_matches = pd.read_csv(RAW_DATA_DIR / "areezvisram12_fixture" / "matches.csv")
df_stages = pd.read_csv(RAW_DATA_DIR / "areezvisram12_fixture" / "tournament_stages.csv")

print(f"Teams   : {df_teams.shape}")
print(f"Matches : {df_matches.shape}")
print(f"Stages  : {df_stages.shape}")

# ── Load team snapshot (Elo + form + cluster) ──────────────────────────────────
df_snapshot = pd.read_parquet(PROCESSED_DATA_DIR / "team_snapshot_clustered.parquet")
print(f"Snapshot: {df_snapshot.shape}")
print(f"\nSnapshot columns: {df_snapshot.columns.tolist()}")

Model loaded. Features: 90
Teams   : (48, 5)
Matches : (104, 8)
Stages  : (7, 3)
Snapshot: (48, 40)

Snapshot columns: ['team', 'group', 'elo', 'form_5_matches_played', 'form_5_win_rate', 'form_5_draw_rate', 'form_5_loss_rate', 'form_5_goals_scored_avg', 'form_5_goals_conceded_avg', 'form_5_goal_diff_avg', 'form_5_clean_sheet_rate', 'form_5_failed_score_rate', 'form_5_points_avg', 'form_5_weighted_points', 'form_10_matches_played', 'form_10_win_rate', 'form_10_draw_rate', 'form_10_loss_rate', 'form_10_goals_scored_avg', 'form_10_goals_conceded_avg', 'form_10_goal_diff_avg', 'form_10_clean_sheet_rate', 'form_10_failed_score_rate', 'form_10_points_avg', 'form_10_weighted_points', 'form_20_matches_played', 'form_20_win_rate', 'form_20_draw_rate', 'form_20_loss_rate', 'form_20_goals_scored_avg', 'form_20_goals_conceded_avg', 'form_20_goal_diff_avg', 'form_20_clean_sheet_rate', 'form_20_failed_score_rate', 'form_20_points_avg', 'form_20_weighted_points', 'cluster', 'pc1', 'pc2', 'cluster_na

In [15]:
# ── Build feature vector lookup per team ──────────────────────────────────────
FORM_WINDOWS = [5, 10, 20]
FORM_STATS = [
    "win_rate", "draw_rate", "loss_rate",
    "goals_scored_avg", "goals_conceded_avg", "goal_diff_avg",
    "clean_sheet_rate", "failed_score_rate", "points_avg",
    "matches_played", "weighted_points",
]

df_master = pd.read_parquet(PROCESSED_DATA_DIR / "master_features.parquet")
SNAPSHOT_DATE = pd.Timestamp("2026-03-31")

# ── Latest FIFA ranking per team ───────────────────────────────────────────────
latest_rankings = (
    df_master[df_master["date"] <= SNAPSHOT_DATE]
    .sort_values("date")
    .groupby("home_team")
    .last()["ranking_home"]
    .reset_index()
    .rename(columns={"home_team": "team", "ranking_home": "ranking"})
)
df_snapshot = df_snapshot.merge(latest_rankings, on="team", how="left")
median_rank = latest_rankings["ranking"].median()
df_snapshot["ranking"] = df_snapshot["ranking"].fillna(median_rank)

# ── Squad value per team ───────────────────────────────────────────────────────
latest_squad_value = (
    df_master[df_master["date"] <= SNAPSHOT_DATE]
    .sort_values("date")
    .groupby("home_team")
    .last()["squad_value_home"]
    .reset_index()
    .rename(columns={"home_team": "team", "squad_value_home": "squad_value"})
)
df_snapshot = df_snapshot.merge(latest_squad_value, on="team", how="left")
median_squad = latest_squad_value["squad_value"].median()
df_snapshot["squad_value"] = df_snapshot["squad_value"].fillna(median_squad)

# ── Encode cluster ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(["Consolidated Mid-Tier", "Dynamic Mid-Tier", "Elite", "Non-WC2026", "Underdogs"])
df_snapshot["cluster_enc"] = le.transform(df_snapshot["cluster_name"])

print("Snapshot enriched with rankings, squad value and cluster encoding.")
print(df_snapshot[["team", "elo", "ranking", "squad_value", "cluster_name"]].head(8).to_string())

# ── Build feature vector function ─────────────────────────────────────────────
def build_match_features(home_team, away_team, neutral=True):
    """
    Build 90-feature vector for a single match prediction.
    neutral=True for all WC matches (played in USA/Canada/Mexico).
    """
    h = df_snapshot[df_snapshot["team"] == home_team].iloc[0]
    a = df_snapshot[df_snapshot["team"] == away_team].iloc[0]

    elo_pre_home  = h["elo"]
    elo_pre_away  = a["elo"]
    elo_diff      = elo_pre_home - elo_pre_away
    win_prob_home = 1 / (1 + 10 ** (-elo_diff / 400))

    features = {
        "elo_pre_home":  elo_pre_home,
        "elo_pre_away":  elo_pre_away,
        "elo_diff":      elo_diff,
        "win_prob_home": win_prob_home,
        "neutral":       int(neutral),
        "ranking_home":  h["ranking"],
        "ranking_away":  a["ranking"],
        "ranking_diff":  h["ranking"] - a["ranking"],
        "squad_value_home": h["squad_value"],
        "squad_value_away": a["squad_value"],
        "squad_value_diff": h["squad_value"] - a["squad_value"],
    }

    # Form features
    for w in FORM_WINDOWS:
        for stat in FORM_STATS:
            features[f"home_form_{w}_{stat}"] = h[f"form_{w}_{stat}"]
            features[f"away_form_{w}_{stat}"] = a[f"form_{w}_{stat}"]

    # H2H — use neutral values (no historical H2H for future matches)
    h2h_defaults = {
        "h2h_matches":             0,
        "h2h_win_rate_a":          0.5,
        "h2h_goal_diff_a":         0.0,
        "h2h_elo_edge_a":          0.0,
        "h2h_weighted_edge_a":     0.0,
        "h2h_decay_weight":        0.0,
        "h2h_reliable":            0,
        "transitive_common_rivals": 0,
        "transitive_edge_a":       0.0,
        "transitive_goal_diff_edge": 0.0,
        "transitive_reliable":     0,
    }
    features.update(h2h_defaults)

    # Cluster
    features["home_cluster_enc"] = h["cluster_enc"]
    features["away_cluster_enc"] = a["cluster_enc"]

    # Return in exact MODEL_FEATURES order
    return np.array([features[f] for f in MODEL_FEATURES], dtype=float)

# ── Sanity check ───────────────────────────────────────────────────────────────
test_vec = build_match_features("Argentina", "France")
assert len(test_vec) == 90, f"Expected 90 features, got {len(test_vec)}"
probs = xgb_model.predict_proba(test_vec.reshape(1, -1))[0]
print(f"\nTest: Argentina vs France")
print(f"  P(away win — France)    : {probs[0]:.3f}")
print(f"  P(draw)                 : {probs[1]:.3f}")
print(f"  P(home win — Argentina) : {probs[2]:.3f}")

Snapshot enriched with rankings, squad value and cluster encoding.
                 team     elo  ranking  squad_value           cluster_name
0              Mexico  1918.0     15.0   20000000.0  Consolidated Mid-Tier
1        South Africa  1668.0     59.0  203000000.0       Dynamic Mid-Tier
2         South Korea  1914.7     22.0   50000000.0       Dynamic Mid-Tier
3             Czechia  1777.5     34.0   37000000.0       Dynamic Mid-Tier
4              Canada  1892.7     48.0   85000000.0  Consolidated Mid-Tier
5  Bosnia-Herzegovina  1635.3     75.0   22000000.0       Dynamic Mid-Tier
6               Qatar  1633.3     35.0  203000000.0              Underdogs
7         Switzerland  1953.9     19.0  201000000.0                  Elite

Test: Argentina vs France
  P(away win — France)    : 0.424
  P(draw)                 : 0.319
  P(home win — Argentina) : 0.257


In [13]:
import json
with open("../../notebooks/05_simulation/05_monte_carlo.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)
print(''.join(nb["cells"][2]["source"]))

# ── Build feature vector lookup per team ──────────────────────────────────────
# Map snapshot columns to model feature names

FORM_WINDOWS = [5, 10, 20]
FORM_STATS = [
    "win_rate", "draw_rate", "loss_rate",
    "goals_scored_avg", "goals_conceded_avg", "goal_diff_avg",
    "clean_sheet_rate", "failed_score_rate", "points_avg",
    "matches_played", "weighted_points",
]

# Load latest FIFA ranking per team from master_features
df_master = pd.read_parquet(PROCESSED_DATA_DIR / "master_features.parquet")
SNAPSHOT_DATE = pd.Timestamp("2026-03-31")

# Get most recent ranking for each WC2026 team
latest_rankings = (
    df_master[df_master["date"] <= SNAPSHOT_DATE]
    .sort_values("date")
    .groupby("home_team")
    .last()["ranking_home"]
    .reset_index()
    .rename(columns={"home_team": "team", "ranking_home": "ranking"})
)

df_snapshot = df_snapshot.merge(latest_rankings, on="team", how="left")
median_rank = latest_rankings["ranking"].median()
df_snapshot["ranking"] = df_snapsho

## 2. Group Stage Simulation

For each of the 12 groups, simulate all 6 matches (round-robin).
Classify top 2 per group + best 8 third-place teams across all groups.

In [4]:
# ── Group stage helpers ────────────────────────────────────────────────────────

def simulate_match(home_team, away_team, neutral=True):
    """Sample match result from model probabilities. Returns (home_pts, away_pts, result)."""
    vec = build_match_features(home_team, away_team, neutral=neutral)
    probs = xgb_model.predict_proba(vec.reshape(1, -1))[0]
    # probs: [P(away win), P(draw), P(home win)]
    result = np.random.choice(["away", "draw", "home"], p=probs)
    if result == "home":
        return 3, 0, "home"
    elif result == "away":
        return 0, 3, "away"
    else:
        return 1, 1, "draw"

def simulate_group(teams):
    """
    Simulate round-robin for a group of 4 teams.
    Returns dict: team -> {pts, gd, gf, w, d, l}
    Goal diff approximated from form features.
    """
    standings = {t: {"pts": 0, "gd": 0, "gf": 0, "w": 0, "d": 0, "l": 0}
                 for t in teams}

    # All 6 matchups (combinations)
    from itertools import combinations
    for home, away in combinations(teams, 2):
        h_pts, a_pts, result = simulate_match(home, away)
        standings[home]["pts"] += h_pts
        standings[away]["pts"] += a_pts

        # Approximate goals from form
        h_snap = df_snapshot[df_snapshot["team"] == home].iloc[0]
        a_snap = df_snapshot[df_snapshot["team"] == away].iloc[0]
        h_gf = max(0, round(np.random.normal(h_snap["form_10_goals_scored_avg"], 1.0)))
        a_gf = max(0, round(np.random.normal(a_snap["form_10_goals_scored_avg"], 1.0)))

        standings[home]["gf"] += h_gf
        standings[away]["gf"] += a_gf
        standings[home]["gd"] += h_gf - a_gf
        standings[away]["gd"] += a_gf - h_gf

        if result == "home":
            standings[home]["w"] += 1
            standings[away]["l"] += 1
        elif result == "away":
            standings[away]["w"] += 1
            standings[home]["l"] += 1
        else:
            standings[home]["d"] += 1
            standings[away]["d"] += 1

    return standings

def rank_group(standings):
    """Sort group standings: pts desc, gd desc, gf desc."""
    return sorted(
        standings.keys(),
        key=lambda t: (standings[t]["pts"], standings[t]["gd"], standings[t]["gf"]),
        reverse=True,
    )

def get_groups():
    """Return dict: group_letter -> [team1, team2, team3, team4]"""
    groups = {}
    for _, row in df_teams.iterrows():
        g = row["group_letter"]
        if g not in groups:
            groups[g] = []
        groups[g].append(row["team_name"])
    return groups

# ── Sanity check ───────────────────────────────────────────────────────────────
groups = get_groups()
print("Groups loaded:")
for g, teams in sorted(groups.items()):
    print(f"  Group {g}: {', '.join(teams)}")

Groups loaded:
  Group A: Mexico, South Africa, South Korea, Czechia
  Group B: Canada, Bosnia-Herzegovina, Qatar, Switzerland
  Group C: Brazil, Morocco, Haiti, Scotland
  Group D: USA, Paraguay, Australia, Turkey
  Group E: Germany, Curacao, Côte d'Ivoire, Ecuador
  Group F: Netherlands, Japan, Sweden, Tunisia
  Group G: Belgium, Egypt, Iran, New Zealand
  Group H: Spain, Cape Verde, Saudi Arabia, Uruguay
  Group I: France, Senegal, Iraq, Norway
  Group J: Argentina, Algeria, Austria, Jordan
  Group K: Portugal, DR Congo, Uzbekistan, Colombia
  Group L: England, Croatia, Ghana, Panama


## 3. Full Tournament Simulation (10,000 runs)

Cascade: Group stage → Best-third selection → R16 → QF → SF → Final.
Knockout draws modeled with 50/50 penalty shootout on draw.

In [23]:
# ── Knockout match ─────────────────────────────────────────────────────────────
def simulate_knockout_fast(team_a, team_b):
    """Simulate knockout match. Draw → penalty shootout (50/50)."""
    _, _, result = simulate_match_fast(team_a, team_b)
    if result == "draw":
        result = "home" if np.random.random() < 0.5 else "away"
    return team_a if result == "home" else team_b

# ── Fast group simulation ──────────────────────────────────────────────────────
def simulate_group_fast(teams):
    standings = {t: {"pts": 0, "gd": 0, "gf": 0} for t in teams}
    h_snap = {t: df_snapshot[df_snapshot["team"] == t].iloc[0] for t in teams}

    for home, away in combinations(teams, 2):
        h_pts, a_pts, result = simulate_match_fast(home, away)
        standings[home]["pts"] += h_pts
        standings[away]["pts"] += a_pts

        h_gf = max(0, round(np.random.normal(h_snap[home]["form_10_goals_scored_avg"], 1.0)))
        a_gf = max(0, round(np.random.normal(h_snap[away]["form_10_goals_scored_avg"], 1.0)))
        standings[home]["gd"] += h_gf - a_gf
        standings[away]["gd"] += a_gf - h_gf
        standings[home]["gf"] += h_gf
        standings[away]["gf"] += a_gf

    return standings

# ── Best third-place selection ─────────────────────────────────────────────────
def select_best_thirds(third_place_teams, third_place_standings):
    return sorted(
        third_place_teams,
        key=lambda t: (
            third_place_standings[t]["pts"],
            third_place_standings[t]["gd"],
            third_place_standings[t]["gf"],
        ),
        reverse=True,
    )[:8]

# ── Resolve third-place team for a bracket slot ────────────────────────────────
def resolve_third(thirds_sorted, groups_allowed, used_thirds):
    """
    Return best available third from allowed groups, not yet used.
    """
    for team in thirds_sorted:
        if team in used_thirds:
            continue
        team_group = df_teams[df_teams["team_name"] == team]["group_letter"].values[0]
        if team_group in groups_allowed:
            used_thirds.add(team)
            return team
    # fallback: first unused
    for team in thirds_sorted:
        if team not in used_thirds:
            used_thirds.add(team)
            return team

# ── R32 bracket assembly (official WC2026 fixture) ────────────────────────────
def assemble_r32(group_winners, group_runners, best_thirds, third_standings):
    w = group_winners
    r = group_runners

    thirds_sorted = sorted(
        best_thirds,
        key=lambda t: (
            third_standings[t]["pts"],
            third_standings[t]["gd"],
            third_standings[t]["gf"],
        ),
        reverse=True,
    )

    used = set()

    r32 = [
        (r["A"],  r["B"]),                                              # 73: 2A vs 2B
        (w["C"],  r["F"]),                                              # 74: 1C vs 2F
        (w["E"],  resolve_third(thirds_sorted, list("ABCDF"), used)),   # 75: 1E vs 3ABCDF
        (w["F"],  r["C"]),                                              # 76: 1F vs 2C
        (r["E"],  r["I"]),                                              # 77: 2E vs 2I
        (w["I"],  resolve_third(thirds_sorted, list("CDFGH"), used)),   # 78: 1I vs 3CDFGH
        (w["A"],  resolve_third(thirds_sorted, list("CEFHI"), used)),   # 79: 1A vs 3CEFHI
        (w["L"],  resolve_third(thirds_sorted, list("EHIJK"), used)),   # 80: 1L vs 3EHIJK
        (w["G"],  resolve_third(thirds_sorted, list("AEHIJ"), used)),   # 81: 1G vs 3AEHIJ
        (w["D"],  resolve_third(thirds_sorted, list("BEFIJ"), used)),   # 82: 1D vs 3BEFIJ
        (w["H"],  r["J"]),                                              # 83: 1H vs 2J
        (r["K"],  r["L"]),                                              # 84: 2K vs 2L
        (w["B"],  resolve_third(thirds_sorted, list("EFGIJ"), used)),   # 85: 1B vs 3EFGIJ
        (r["D"],  r["G"]),                                              # 86: 2D vs 2G
        (w["J"],  r["H"]),                                              # 87: 1J vs 2H
        (w["K"],  resolve_third(thirds_sorted, list("DEIJL"), used)),   # 88: 1K vs 3DEIJL
    ]
    return r32

# ── Full tournament simulation ─────────────────────────────────────────────────
def simulate_tournament_fast():
    groups = get_groups()
    group_winners   = {}
    group_runners   = {}
    third_teams     = []
    third_standings = {}

    # Group stage
    for letter, teams in sorted(groups.items()):
        standings = simulate_group_fast(teams)
        ranked    = rank_group(standings)
        group_winners[letter] = ranked[0]
        group_runners[letter] = ranked[1]
        third_teams.append(ranked[2])
        third_standings[ranked[2]] = standings[ranked[2]]

    best_thirds = select_best_thirds(third_teams, third_standings)

    phase = {t: "Group" for t in all_teams}
    for t in (list(group_winners.values()) +
              list(group_runners.values()) +
              best_thirds):
        phase[t] = "R32"

    # R32 — 32 teams, 16 matches
    r32_matches = assemble_r32(group_winners, group_runners, best_thirds, third_standings)
    assert len(r32_matches) == 16, f"Expected 16 R32 matches, got {len(r32_matches)}"
    r32_winners = [simulate_knockout_fast(h, a) for h, a in r32_matches]
    for t in r32_winners:
        phase[t] = "R16"

    # R16 — 16 teams, 8 matches
    # Bracket follows stage 3: W73vsW75, W74vsW77, W76vsW78, W79vsW80,
    #                          W83vsW84, W81vsW82, W86vsW88, W85vsW87
    r16_pairs = [
        (r32_winners[0],  r32_winners[2]),   # W73 vs W75
        (r32_winners[1],  r32_winners[4]),   # W74 vs W77
        (r32_winners[3],  r32_winners[5]),   # W76 vs W78
        (r32_winners[6],  r32_winners[7]),   # W79 vs W80
        (r32_winners[10], r32_winners[11]),  # W83 vs W84
        (r32_winners[8],  r32_winners[9]),   # W81 vs W82
        (r32_winners[13], r32_winners[15]),  # W86 vs W88
        (r32_winners[12], r32_winners[14]),  # W85 vs W87
    ]
    r16_winners = [simulate_knockout_fast(h, a) for h, a in r16_pairs]
    for t in r16_winners:
        phase[t] = "QF"

    # QF — 8 teams, 4 matches
    # Bracket follows stage 4: W89vsW90, W93vsW94, W91vsW92, W95vsW96
    qf_pairs = [
        (r16_winners[0], r16_winners[1]),  # W89 vs W90
        (r16_winners[4], r16_winners[5]),  # W93 vs W94
        (r16_winners[2], r16_winners[3]),  # W91 vs W92
        (r16_winners[6], r16_winners[7]),  # W95 vs W96
    ]
    qf_winners = [simulate_knockout_fast(h, a) for h, a in qf_pairs]
    for t in qf_winners:
        phase[t] = "SF"

    # SF — 4 teams, 2 matches
    # Bracket follows stage 5: W97vsW98, W99vsW100
    sf_losers  = []
    sf_winners = []
    sf_pairs = [
        (qf_winners[0], qf_winners[1]),  # W97 vs W98
        (qf_winners[2], qf_winners[3]),  # W99 vs W100
    ]
    for a, b in sf_pairs:
        winner = simulate_knockout_fast(a, b)
        loser  = b if winner == a else a
        sf_winners.append(winner)
        sf_losers.append(loser)
    for t in sf_winners:
        phase[t] = "Final"

    # 3rd place match — stage 6: RU101 vs RU102
    third = simulate_knockout_fast(sf_losers[0], sf_losers[1])
    phase[third] = "Third"

    # Final — stage 7: W101 vs W102
    champion = simulate_knockout_fast(sf_winners[0], sf_winners[1])
    phase[champion] = "Champion"

    return phase

In [22]:
np.random.seed(42)
result = simulate_tournament_fast()
phase_counts = Counter(result.values())
print("Phase distribution in 1 simulation:")
for phase in ["Champion", "Final", "Third", "SF", "QF", "R16", "R32", "Group"]:
    print(f"  {phase}: {phase_counts.get(phase, 0)}")

champions = [t for t, p in result.items() if p == "Champion"]
thirds    = [t for t, p in result.items() if p == "Third"]
finalists = [t for t, p in result.items() if p == "Final"]
print(f"\nChampion : {champions}")
print(f"Finalist : {finalists}")
print(f"3rd place: {thirds}")

Phase distribution in 1 simulation:
  Champion: 1
  Final: 1
  Third: 1
  SF: 1
  QF: 4
  R16: 8
  R32: 16
  Group: 16

Champion : ['Jordan']
Finalist : ['Ghana']
3rd place: ['South Korea']


In [24]:
# ── Run 10,000 simulations ─────────────────────────────────────────────────────
N_SIMS = 10_000
PHASES = ["Group", "R32", "R16", "QF", "SF", "Third", "Final", "Champion"]
counts = {t: {p: 0 for p in PHASES} for t in all_teams}

print(f"Running {N_SIMS:,} simulations...")
for _ in tqdm(range(N_SIMS)):
    result = simulate_tournament_fast()
    for team, phase in result.items():
        if team in counts and phase in counts[team]:
            counts[team][phase] += 1

print("Done.")

Running 10,000 simulations...


100%|██████████| 10000/10000 [15:52<00:00, 10.50it/s] 


Done.


In [25]:
rows = []
for team in all_teams:
    group   = df_teams[df_teams["team_name"] == team]["group_letter"].values[0]
    elo     = df_snapshot[df_snapshot["team"] == team]["elo"].values[0]
    cluster = df_snapshot[df_snapshot["team"] == team]["cluster_name"].values[0]

    p_r32      = sum(counts[team][p] for p in
                     ["R32","R16","QF","SF","Third","Final","Champion"]) / N_SIMS
    p_r16      = sum(counts[team][p] for p in
                     ["R16","QF","SF","Third","Final","Champion"]) / N_SIMS
    p_qf       = sum(counts[team][p] for p in
                     ["QF","SF","Third","Final","Champion"]) / N_SIMS
    p_sf       = sum(counts[team][p] for p in
                     ["SF","Third","Final","Champion"]) / N_SIMS
    p_final    = sum(counts[team][p] for p in
                     ["Final","Champion"]) / N_SIMS
    p_champion =  counts[team]["Champion"] / N_SIMS

    rows.append({
        "team":       team,
        "group":      group,
        "elo":        round(elo, 0),
        "cluster":    cluster,
        "p_r32":      p_r32,
        "p_r16":      p_r16,
        "p_qf":       p_qf,
        "p_sf":       p_sf,
        "p_final":    p_final,
        "p_champion": p_champion,
    })

df_results = pd.DataFrame(rows).sort_values("p_champion", ascending=False).reset_index(drop=True)

pd.set_option("display.float_format", "{:.1%}".format)
print("=" * 95)
print("FIFA WORLD CUP 2026 — TOURNAMENT WINNER PROBABILITIES (10,000 simulations)")
print("=" * 95)
print(f"{'#':<4} {'Team':<25} {'Group':<7} {'Elo':<7} {'R32':>6} {'R16':>6} "
      f"{'QF':>6} {'SF':>6} {'Final':>6} {'Win':>7}")
print("-" * 95)
for i, row in df_results.iterrows():
    print(f"{i+1:<4} {row['team']:<25} {row['group']:<7} {row['elo']:<7.0f} "
          f"{row['p_r32']:>6.1%} {row['p_r16']:>6.1%} {row['p_qf']:>6.1%} "
          f"{row['p_sf']:>6.1%} {row['p_final']:>6.1%} {row['p_champion']:>7.1%}")

pd.reset_option("display.float_format")
output_path = Path("../..").resolve() / "outputs" / "predictions" / "tournament_probabilities.csv"
df_results.to_csv(output_path, index=False)
print(f"\nSaved → {output_path}")

FIFA WORLD CUP 2026 — TOURNAMENT WINNER PROBABILITIES (10,000 simulations)
#    Team                      Group   Elo        R32    R16     QF     SF  Final     Win
-----------------------------------------------------------------------------------------------
1    Argentina                 J       2153     67.6%  37.9%  21.2%  12.8%   7.7%    4.3%
2    Croatia                   L       1976     73.2%  41.0%  25.8%  13.5%   7.7%    4.2%
3    Spain                     H       2195     73.7%  40.5%  22.0%  11.8%   6.7%    3.6%
4    England                   L       2090     68.0%  36.6%  21.3%  11.3%   6.1%    3.5%
5    France                    I       2114     67.2%  39.8%  24.5%  14.2%   7.0%    3.5%
6    Colombia                  K       2032     79.7%  37.2%  21.4%   9.8%   5.4%    2.9%
7    Switzerland               B       1954     86.4%  42.5%  19.1%   9.8%   5.6%    2.8%
8    Portugal                  K       2000     64.3%  30.7%  18.9%   8.8%   4.8%    2.8%
9    Australia     